# 00 — Monitor Oracle IoT  
Dashboard interativo equivalente ao Browse do phpMyAdmin.

- **Refresh manual** — botão 🔄  
- **Auto-refresh** — toggle com intervalo configurável  
- **Timestamp** da última atualização  
- **Últimos N registros** — slider de quantidade  

> Execute as células em ordem (`Run All`). O painel aparece na Célula 2.

In [5]:
# =============================================================================
# Célula 0 — Configuração e teste de conexão
# =============================================================================
import os
import threading
from datetime import datetime

import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output
from sqlalchemy import create_engine, text

# --- Credenciais: usa variáveis de ambiente se definidas, senão usa dersao ---
ORACLE_USER     = os.environ.get('ORACLE_USER',         'dersao')
ORACLE_PASSWORD = os.environ.get('ORACLE_PASSWORD',     '986960440')
ORACLE_HOST     = os.environ.get('ORACLE_HOST',         'localhost')
ORACLE_PORT     = os.environ.get('ORACLE_PORT',         '1521')
ORACLE_SERVICE  = os.environ.get('ORACLE_SERVICE_NAME', 'xepdb1')

# IMPORTANTE: SQLAlchemy trata "host:port/nome" como SID (legado).
# Para SERVICE_NAME obrigatoriamente usar "?service_name=..."
ORACLE_CONN_STR = (
    f'oracle+oracledb://{ORACLE_USER}:{ORACLE_PASSWORD}'
    f'@{ORACLE_HOST}:{ORACLE_PORT}/?service_name={ORACLE_SERVICE}'
)

_engine = create_engine(ORACLE_CONN_STR)

def _read_sql(query, conn):
    """pd.read_sql via SQLAlchemy. Normaliza colunas para minusculo (Oracle retorna MAIUSCULO)."""
    df = pd.read_sql(text(query), conn)
    df.columns = df.columns.str.lower()
    return df

# Teste de conexão
try:
    with _engine.connect() as _conn:
        _total = _conn.execute(text("SELECT COUNT(*) FROM sensor_data")).fetchone()[0]
    print(f'Conexao Oracle OK  |  sensor_data: {_total:,} registros')
    print(f'Host: {ORACLE_HOST}:{ORACLE_PORT}  Service: {ORACLE_SERVICE}  Usuario: {ORACLE_USER}')
except Exception as _e:
    print(f'ERRO de conexao: {_e}')
    print('Verifique se OracleServiceXE e TNSListener estao rodando.')


Conexao Oracle OK  |  sensor_data: 35 registros
Host: localhost:1521  Service: xepdb1  Usuario: dersao


In [6]:
# =============================================================================
# Célula 1 — Dashboard interativo (Refresh + Auto-refresh)
# =============================================================================
from IPython.display import HTML

# CSS: toolbar fixa no topo enquanto rola a página
display(HTML("""
<style>
.sticky-refresh {
    position: sticky !important;
    top: 0 !important;
    z-index: 1000 !important;
    background-color: #111827 !important;
    padding: 6px 4px !important;
    border-bottom: 1px solid #2d3748 !important;
    margin-bottom: 6px !important;
}
</style>
"""))

# ---------- Controles ----------
btn_refresh = widgets.Button(
    description='Refresh',
    button_style='primary',
    icon='refresh',
    layout=widgets.Layout(width='110px', height='32px'),
)
toggle_auto = widgets.ToggleButton(
    value=False,
    description='Auto OFF',
    button_style='',
    icon='clock-o',
    layout=widgets.Layout(width='110px', height='32px'),
)
dropdown_interval = widgets.Dropdown(
    options=[('5s', 5), ('10s', 10), ('30s', 30), ('60s', 60)],
    value=5,
    description='Intervalo:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='150px'),
    disabled=True,
)
slider_rows = widgets.IntSlider(
    value=30, min=10, max=200, step=10,
    description='Linhas:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='320px'),
)
dropdown_collection = widgets.Dropdown(
    options=['(todas)'],
    value='(todas)',
    description='Collection:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='340px'),
)
lbl_status = widgets.HTML(value='<span style="color:#888">Aguardando primeiro refresh...</span>')
out_main   = widgets.Output()

# ---------- Estado interno ----------
_stop_event  = threading.Event()
_auto_thread = None
_SEP         = '-' * 70

# ---------- Lógica de dados ----------

def _fetch_collections():
    try:
        with _engine.connect() as conn:
            rows = conn.execute(text("""
                SELECT collection_id
                FROM sensor_data
                GROUP BY collection_id
                ORDER BY MAX(ts_epoch) DESC
            """)).fetchall()
        return ['(todas)'] + [r[0] for r in rows]
    except Exception:
        return ['(todas)']

def _fetch_all(n_rows, collection):
    where = (
        ""
        if collection == '(todas)'
        else "WHERE collection_id = '" + collection + "'"
    )

    with _engine.connect() as conn:

        df_stats = _read_sql(
            """
            SELECT
                COUNT(*)                                        AS total_registros,
                MAX(id)                                         AS ultimo_id,
                ROUND(AVG(sample_rate), 1)                     AS hz_medio,
                ROUND((MAX(ts_epoch) - MIN(ts_epoch)) / 60, 1) AS duracao_total_min,
                ROUND(
                    ((CAST(SYS_EXTRACT_UTC(SYSTIMESTAMP) AS DATE)
                      - DATE '1970-01-01') * 86400)
                    - MAX(ts_epoch)
                , 1)                                            AS ultimo_envio_ha_s,
                COUNT(DISTINCT collection_id)                   AS num_colecoes,
                COUNT(DISTINCT device_id)                       AS num_dispositivos
            FROM sensor_data
            """ + where,
            conn,
        )

        df_dist = _read_sql(
            """
            SELECT
                NVL(cmd_speed_label, fan_state)              AS velocidade,
                NVL(rot_state_label, '-')                    AS rotacao,
                COUNT(*)                                     AS registros,
                ROUND(COUNT(*) * 100.0
                      / SUM(COUNT(*)) OVER (), 1)            AS pct
            FROM sensor_data
            """
            + where
            + """
            GROUP BY NVL(cmd_speed_label, fan_state),
                     NVL(rot_state_label, '-')
            ORDER BY registros DESC
            """,
            conn,
        )

        # ts_epoch em UTC → subtrai 10800s (3h) para converter para horário de Brasília (BRT = UTC-3)
        # Brasil aboliu horário de verão em 2019, BRT = UTC-3 permanente
        df_last = _read_sql(
            """
            SELECT * FROM (
                SELECT
                    id,
                    TO_CHAR(
                        TIMESTAMP '1970-01-01 00:00:00 UTC'
                        + NUMTODSINTERVAL(ts_epoch - 10800, 'SECOND'),
                        'DD/MM HH24:MI:SS'
                    )                    AS data_hora,
                    ROUND(accel_x_g,  4) AS ax_g,
                    ROUND(accel_y_g,  4) AS ay_g,
                    ROUND(accel_z_g,  4) AS az_g,
                    ROUND(gyro_x_dps, 2) AS gx_dps,
                    ROUND(gyro_y_dps, 2) AS gy_dps,
                    ROUND(gyro_z_dps, 2) AS gz_dps,
                    ROUND(temperature,2) AS temp_c,
                    fan_state,
                    cmd_speed_label      AS velocidade,
                    rot_state_label      AS rotacao,
                    sample_rate          AS hz,
                    rssi                 AS rssi_dbm,
                    collection_id
                FROM sensor_data
                """
            + where
            + " ORDER BY id DESC ) WHERE ROWNUM <= " + str(n_rows),
            conn,
        )

    return df_stats, df_dist, df_last

def _render_table(df, title):
    print('\n' + _SEP)
    print('  ' + title)
    print(_SEP)
    display(
        df.style
        .set_properties(**{'font-size': '12px', 'text-align': 'right'})
        .set_table_styles([{
            'selector': 'th',
            'props': [('font-size', '12px'), ('text-align', 'center'),
                      ('background-color', '#1a1a2e'), ('color', 'white')]
        }])
        .hide(axis='index')
        .format(na_rep='-')
    )

def _do_refresh():
    n_rows     = slider_rows.value
    collection = dropdown_collection.value
    ts         = datetime.now().strftime('%H:%M:%S')
    with out_main:
        clear_output(wait=True)
        try:
            df_stats, df_dist, df_last = _fetch_all(n_rows, collection)
            total    = int(df_stats['total_registros'].iloc[0])
            ultimo_s = df_stats['ultimo_envio_ha_s'].iloc[0]
            lbl_status.value = (
                '<span style="color:#00c896">'
                'Ultima atualizacao: <b>' + ts + '</b> &nbsp;|&nbsp; '
                'Total: <b>' + f'{total:,}' + '</b> registros &nbsp;|&nbsp; '
                'Ultimo envio ha <b>' + str(ultimo_s) + 's</b>'
                '</span>'
            )
            _render_table(df_stats, 'RESUMO GERAL')
            _render_table(df_dist,  'DISTRIBUICAO POR CLASSE')
            _render_table(df_last,  'ULTIMOS ' + str(n_rows) + ' REGISTROS (mais recente primeiro)')
        except Exception as e:
            lbl_status.value = '<span style="color:#ff4444">Erro: ' + str(e) + '</span>'
            print('Erro ao carregar dados: ' + str(e))

def _auto_loop():
    while not _stop_event.is_set():
        _do_refresh()
        _stop_event.wait(timeout=dropdown_interval.value)

# ---------- Eventos ----------
def on_refresh(_):
    _do_refresh()

def on_toggle_auto(change):
    global _auto_thread
    if change['new']:
        toggle_auto.description    = 'Auto ON'
        toggle_auto.button_style   = 'success'
        dropdown_interval.disabled = False
        _stop_event.clear()
        _auto_thread = threading.Thread(target=_auto_loop, daemon=True)
        _auto_thread.start()
    else:
        toggle_auto.description    = 'Auto OFF'
        toggle_auto.button_style   = ''
        dropdown_interval.disabled = True
        _stop_event.set()

btn_refresh.on_click(on_refresh)
toggle_auto.observe(on_toggle_auto, names='value')

# ---------- Inicialização ----------
dropdown_collection.options = _fetch_collections()

toolbar = widgets.VBox([
    widgets.HBox(
        [btn_refresh, toggle_auto, dropdown_interval, dropdown_collection],
        layout=widgets.Layout(gap='10px', align_items='center'),
    ),
    widgets.HBox([slider_rows]),
    lbl_status,
])
toolbar.add_class('sticky-refresh')

display(toolbar)
display(out_main)
_do_refresh()


Output()

In [7]:
# =============================================================================
# Célula 2 — Balanço de coleta por classe (guia para treinamento)
# =============================================================================
TARGET_AMOSTRAS = 500   # meta mínima por classe para treinamento

with _engine.connect() as _conn:
    df_bal = _read_sql(f"""
        SELECT
            collection_id,
            NVL(cmd_speed_label, fan_state)          AS classe,
            COUNT(*)                                  AS amostras,
            SUM(CASE WHEN transition_marker = 1
                     THEN 1 ELSE 0 END)               AS transicao,
            SUM(CASE WHEN transition_marker = 0
                     THEN 1 ELSE 0 END)               AS validos,
            ROUND(MIN(
                TIMESTAMP '1970-01-01 00:00:00 UTC'
                + NUMTODSINTERVAL(ts_epoch - 10800, 'SECOND')
            ), 0)                                     AS inicio,
            ROUND((MAX(ts_epoch) - MIN(ts_epoch)) / 60, 1) AS duracao_min
        FROM sensor_data
        WHERE transition_marker IS NOT NULL
        GROUP BY collection_id, NVL(cmd_speed_label, fan_state)
        ORDER BY collection_id, classe
    """, _conn)

def _status(validos):
    v = int(validos) if validos else 0
    if v >= TARGET_AMOSTRAS:
        return f'OK ({v})'
    return f'faltam {TARGET_AMOSTRAS - v}'

df_bal['meta'] = df_bal['validos'].apply(_status)

print('=' * 70)
print(f'  BALANCO DE COLETA — meta: {TARGET_AMOSTRAS} amostras validas por classe')
print('=' * 70)
display(
    df_bal.style
    .set_properties(**{'font-size': '12px', 'text-align': 'right'})
    .set_table_styles([{
        'selector': 'th',
        'props': [('font-size', '12px'), ('text-align', 'center'),
                  ('background-color', '#1a1a2e'), ('color', 'white')]
    }])
    .applymap(lambda v: 'color: #00c896' if str(v).startswith('OK') else
                        ('color: #ff4444' if str(v).startswith('faltam') else ''),
              subset=['meta'])
    .hide(axis='index')
    .format(na_rep='-')
)


DatabaseError: (oracledb.exceptions.DatabaseError) ORA-00932: tipos de dados inconsistentes: esperava DATE UNIT obteve NUMBER
Help: https://docs.oracle.com/error-help/db/ora-00932/
[SQL: 
        SELECT
            collection_id,
            NVL(cmd_speed_label, fan_state)          AS classe,
            COUNT(*)                                  AS amostras,
            SUM(CASE WHEN transition_marker = 1
                     THEN 1 ELSE 0 END)               AS transicao,
            SUM(CASE WHEN transition_marker = 0
                     THEN 1 ELSE 0 END)               AS validos,
            ROUND(MIN(
                TIMESTAMP '1970-01-01 00:00:00 UTC'
                + NUMTODSINTERVAL(ts_epoch - 10800, 'SECOND')
            ), 0)                                     AS inicio,
            ROUND((MAX(ts_epoch) - MIN(ts_epoch)) / 60, 1) AS duracao_min
        FROM sensor_data
        WHERE transition_marker IS NOT NULL
        GROUP BY collection_id, NVL(cmd_speed_label, fan_state)
